# Axis 3 (knowledge): why it rises then falls
Trace: `results/mirt_humanprior_lineageprior_floors/`. The story per vendor lineage, with the benchmark scores that drive it.

In [1]:

import sys, re
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "fit.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import xarray as xr
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from data import load_eci_data, load_benchmark_floors
from analysis.rotation import canonicalize_factors, nonneg_rotate
from analysis.timelines import mirt_model_timeline_df
from analysis.stats import _release_dates

TRACE = ROOT / "results/mirt_humanprior_lineageprior_floors/trace_mirt_k3_humanprior_lineageprior_floors.nc"
PLOTS = ROOT / "plots/axis3_instrument_swap"
PLOTS.mkdir(parents=True, exist_ok=True)
THIN = 5

C = dict(blue="#0072B2", sky="#56B4E9", orange="#E69F00", verm="#D55E00",
         green="#009E73", pink="#CC79A7", gray="#999999", dark="#333333")

def show(fig, name):
    fig.update_layout(template="plotly_white", title=dict(x=0.5, font=dict(size=13)))
    fig.write_image(PLOTS / f"{name}.png", scale=2)
    fig.show()

# ── trace → displayed (nonneg-rotated) abilities ─────────────────────────────
post = xr.open_dataset(TRACE, group="posterior")
A = post["A"].isel(draw=slice(None, None, THIN)).values
theta = post["theta"].isel(draw=slice(None, None, THIN)).values
tau = post["tau_A"].isel(draw=slice(None, None, THIN)).values
drift_draws = post["lin_drift"].isel(draw=slice(None, None, THIN)).values
post.close()
S = A.shape[0] * A.shape[1]
A, theta, tau = (A.reshape(S, *A.shape[2:]), theta.reshape(S, *theta.shape[2:]),
                 tau.reshape(S, -1))
key = (A ** 2).sum(axis=1); order0 = np.argsort(-key, axis=1)
drift_canon = np.take_along_axis(drift_draws.reshape(S, -1), order0, axis=1)
A, theta, tau = canonicalize_factors(A, theta, tau, rank_track=True)
Tl, Tt, Phi = nonneg_rotate(A.mean(0))
A_rot, theta_rot = A @ Tl, theta @ Tt
signs = np.where(A_rot.mean(0).sum(0) < 0, -1.0, 1.0)
A_rot *= signs; theta_rot *= signs
order = np.argsort(-(A_rot.mean(0) ** 2).sum(0))
A_rot, theta_rot = A_rot[:, :, order], theta_rot[:, :, order]
DRIFT3 = drift_canon[:, order][:, 2].mean()

# ── data ─────────────────────────────────────────────────────────────────────
data = load_eci_data(include_all_benchmarks=True)
raw = pd.read_csv(ROOT / "data/processed/benchmarks_merged.csv")
models = data.mlookup.sort_values("model_idx")["model"].tolist()
benches = data.blookup.sort_values("benchmark_idx")["benchmark"].tolist()
mdates, _ = _release_dates(raw)
med_load = pd.Series(np.median(A_rot[:, :, 2], axis=0), index=benches)

obs = pd.DataFrame({"model": [models[i] for i in data.model_idx],
                    "benchmark": [benches[i] for i in data.bench_idx],
                    "score": data.scores})
obs["mdate"] = pd.to_datetime(obs["model"].map(mdates))

summ = pd.DataFrame({"model": models,
                     "date": pd.to_datetime([mdates.get(m) for m in models]),
                     "ax1": theta_rot[:, :, 0].mean(0),
                     "ax3": theta_rot[:, :, 2].mean(0),
                     "ax3_sd": theta_rot[:, :, 2].std(0)}).set_index("model")

tl = mirt_model_timeline_df(theta_rot, 2, data, raw)          # informed θ₃ timeline
tl["q"] = tl.release_date.dt.to_period("Q")

# ── lineage nodes: pool effort/token variants to the release node ────────────
LIN = pd.read_csv(ROOT / "data/curated/lineage_map.csv").drop_duplicates("raw_string")
LIN = LIN[LIN.in_chain.astype(str).str.lower() == "yes"]
node_of = {r.raw_string: (r.chain, r.node) for r in LIN.itertuples() if r.raw_string in models}

# representative-node lists (curated); pooling grabs every variant of each node
CHAINS = {
    "OpenAI gpt": [("5", "gpt-5-2025-08-07_medium"), ("5.1", "gpt-5.1-2025-11-13_medium"),
                   ("5.2", "gpt-5.2-2025-12-11_high"), ("5.4", "gpt-5.4-2026-03-05_medium"),
                   ("5.5", "gpt-5.5_unknown"), ("5.6", "gpt-5.6-sol_max")],
    "Anthropic sonnet": [("3.5", "claude-3-5-sonnet-20241022"),
                         ("4", "claude-sonnet-4-20250514_unknown"),
                         ("4.5", "claude-sonnet-4-5-20250929_unknown"),
                         ("4.6", "claude-sonnet-4-6_unknown"), ("5", "claude-sonnet-5_max")],
    "Anthropic opus → Fable": [("4.1", "claude-opus-4-1-20250805_unknown"),
                               ("4.5", "claude-opus-4-5-20251101_unknown"),
                               ("4.6", "claude-opus-4-6"), ("4.7", "claude-opus-4-7_unknown"),
                               ("4.8", "claude-opus-4-8_max"), ("Fable 5", "claude-fable-5_max")],
}
def node_members(rep):
    cn = node_of.get(rep)
    return [m for m in models if node_of.get(m) == cn] if cn else [rep]

def node_score(rep, bench):
    ms = node_members(rep)
    v = obs[(obs.model.isin(ms)) & (obs.benchmark == bench)]["score"]
    return v.mean() if len(v) else None

def node_theta(rep, col):
    ms = [m for m in node_members(rep) if m in summ.index]
    return summ.loc[ms, col].mean() if ms else None

KNOW = ["SimpleBench", "SimpleQA Verified"]     # warm = knowledge
REAS = "ARC-AGI-2"                              # blue = reasoning
print(f"{S} draws · lineage drift axis-3 = +{DRIFT3:.3f}/release")


16000 draws · lineage drift axis-3 = +0.036/release


### 1 · The phenomenon: θ₃ climbs, then falls after mid-2025

In [2]:

fig = go.Figure()
fig.add_trace(go.Scatter(x=tl.release_date, y=tl["mean"], mode="markers", name="AI models",
                         marker=dict(color=C["sky"], size=6, opacity=0.6), text=tl.name,
                         hovertemplate="<b>%{text}</b><br>%{x|%Y-%m-%d} · θ₃ %{y:.2f}<extra></extra>"))
qm = tl.groupby("q")["mean"].median()
fig.add_trace(go.Scatter(x=qm.index.to_timestamp() + pd.offsets.Day(45), y=qm.values,
                         mode="lines", name="quarterly median", line=dict(color=C["dark"], width=3)))
pk = qm.idxmax()
fig.add_annotation(x=(pk.to_timestamp() + pd.offsets.Day(45)).isoformat(), y=float(qm.max()),
                   text=f"peak {qm.max():.2f} ({pk})", arrowhead=2, ax=0, ay=-42,
                   font=dict(color=C["verm"], size=12), arrowcolor=C["verm"])
lo = qm.loc["2025Q3":].idxmin()
fig.add_annotation(x=(lo.to_timestamp() + pd.offsets.Day(45)).isoformat(), y=float(qm.loc[lo]),
                   text=f"{qm.loc[lo]:.2f} ({lo})", arrowhead=2, ax=0, ay=48,
                   font=dict(color=C["dark"], size=12), arrowcolor=C["dark"])
fig.update_layout(title="Axis-3 (knowledge) ability over time",
                  yaxis_title="knowledge ability θ₃",
                  height=470, width=920, legend=dict(orientation="h", y=1.03))
show(fig, "p1_phenomenon")


### 2 · The cause, per lineage.
**Top:** raw benchmark scores along the chain — reasoning (blue) rises faster than knowledge (warm). **Bottom:** the skills the model infers — reasoning θ₁ rises, so the model expects knowledge to rise too; flat knowledge scores against that rising expectation force knowledge θ₃ **down**.

In [3]:

fig = make_subplots(rows=2, cols=3, shared_xaxes=True,
                    row_heights=[0.5, 0.5], vertical_spacing=0.08, horizontal_spacing=0.06,
                    subplot_titles=list(CHAINS) + ["", "", ""])
for j, (panel, nodes) in enumerate(CHAINS.items(), 1):
    labs = [lab for lab, _ in nodes]
    reps = [rep for _, rep in nodes]
    # top: benchmark scores
    fig.add_trace(go.Scatter(x=labs, y=[node_score(r, REAS) for r in reps],
                             mode="lines+markers", name="ARC-AGI-2 (reasoning)",
                             line=dict(color=C["blue"], width=2), marker=dict(size=8),
                             connectgaps=True, showlegend=(j == 1)), row=1, col=j)
    for b, col in zip(KNOW, [C["verm"], C["orange"]]):
        fig.add_trace(go.Scatter(x=labs, y=[node_score(r, b) for r in reps],
                                 mode="lines+markers", name=f"{b} (knowledge)",
                                 line=dict(color=col, width=2), marker=dict(size=8),
                                 connectgaps=True, showlegend=(j == 1)), row=1, col=j)
    # bottom: inferred skills
    def _lbl(v):
        return [f"{v[0]:.2f}"] + [""] * (len(v) - 2) + [f"{v[-1]:.2f}"]
    y1 = [node_theta(r, "ax1") for r in reps]
    y3 = [node_theta(r, "ax3") for r in reps]
    fig.add_trace(go.Scatter(x=labs, y=y1, mode="lines+markers+text", text=_lbl(y1),
                             textposition="top center", textfont=dict(size=10, color=C["blue"]),
                             name="θ₁ reasoning skill",
                             line=dict(color=C["blue"], width=2, dash="dot"),
                             marker=dict(size=8, symbol="square"), showlegend=(j == 1)),
                  row=2, col=j)
    fig.add_trace(go.Scatter(x=labs, y=y3, mode="lines+markers+text", text=_lbl(y3),
                             textposition="bottom center", textfont=dict(size=10, color=C["verm"]),
                             name="θ₃ knowledge skill",
                             line=dict(color=C["verm"], width=2, dash="dot"),
                             marker=dict(size=8, symbol="square"), showlegend=(j == 1)),
                  row=2, col=j)
    fig.update_yaxes(range=[0, 1], row=1, col=j, title="score" if j == 1 else None)
    fig.update_yaxes(range=[0, 2.7], row=2, col=j, title="ability θ" if j == 1 else None)
fig.update_layout(title="Reasoning scores outrun knowledge scores ⇒ knowledge skill falls",
                  height=680, width=1080, margin=dict(t=90),
                  legend=dict(orientation="h", y=-0.10, font=dict(size=12)))
show(fig, "p2_per_lineage_cause")


### 3 · Why the lineage prior can't stop it.
Triangle = the prior's only opinion (previous node + 0.04). Where a node has real knowledge scores the posterior leaves the triangle downward; where it has none it just sits on the triangle. The prior chains to the *previous* node, never the peak, and +0.04 is dwarfed by one benchmark score.

In [4]:

fig = make_subplots(rows=1, cols=3, subplot_titles=list(CHAINS), shared_yaxes=True,
                    horizontal_spacing=0.05)
for j, (panel, nodes) in enumerate(CHAINS.items(), 1):
    labs = [lab for lab, _ in nodes]
    th = [node_theta(rep, "ax3") for _, rep in nodes]
    sd = [summ.loc[[m for m in node_members(rep) if m in summ.index], "ax3_sd"].mean()
          for _, rep in nodes]
    ghost = [None] + [th[i - 1] + DRIFT3 for i in range(1, len(nodes))]
    fig.add_trace(go.Scatter(x=labs, y=th, error_y=dict(array=sd, width=2, thickness=1),
                             mode="lines+markers", name="posterior θ₃",
                             marker=dict(color=C["verm"], size=9),
                             line=dict(color=C["verm"], width=2), showlegend=(j == 1)),
                  row=1, col=j)
    seg_x, seg_y = [], []
    for i in range(1, len(nodes)):
        seg_x += [labs[i], labs[i], None]
        seg_y += [ghost[i], th[i], None]
    fig.add_trace(go.Scatter(x=seg_x, y=seg_y, mode="lines",
                             name="pull of the data",
                             line=dict(color=C["gray"], width=1.5, dash="dot"),
                             showlegend=(j == 1), hoverinfo="skip"), row=1, col=j)
    fig.add_trace(go.Scatter(x=labs, y=ghost, mode="markers",
                             name=f"prior alone (previous + {DRIFT3:.2f})",
                             marker=dict(symbol="triangle-up-open", size=13,
                                         color=C["dark"], line_width=1.5),
                             showlegend=(j == 1)), row=1, col=j)
fig.update_yaxes(title="knowledge θ₃", row=1, col=1)
fig.update_layout(title="Posterior vs the prior's expectation", height=460, width=1050,
                  legend=dict(orientation="h", y=-0.16, font=dict(size=12)))
show(fig, "p3_prior_vs_data")


### Ledger (node-pooled)

In [5]:

rows = []
for panel, nodes in CHAINS.items():
    prev = None
    for lab, rep in nodes:
        t1, t3 = node_theta(rep, "ax1"), node_theta(rep, "ax3")
        rows.append({"chain": panel, "node": lab,
                     "θ1 reasoning": round(t1, 2), "θ3 knowledge": round(t3, 2),
                     "Δθ3": (round(t3 - prev, 2) if prev is not None else None),
                     "prior Δ": round(DRIFT3, 2),
                     "ARC-AGI-2": (round(node_score(rep, REAS), 2)
                                   if node_score(rep, REAS) is not None else None),
                     "SimpleBench": (round(node_score(rep, "SimpleBench"), 2)
                                     if node_score(rep, "SimpleBench") is not None else None),
                     "SimpleQA": (round(node_score(rep, "SimpleQA Verified"), 2)
                                  if node_score(rep, "SimpleQA Verified") is not None else None)})
        prev = t3
pd.DataFrame(rows)


,chain,node,θ1 reasoning,θ3 knowledge,Δθ3,prior Δ,ARC-AGI-2,SimpleBench,SimpleQA
0,OpenAI gpt,5,0.64,1.58,NaN,0.04,0.05,0.57,0.51
1,OpenAI gpt,5.1,0.72,1.16,-0.42,0.04,0.07,0.53,0.49
2,OpenAI gpt,5.2,1.20,0.79,-0.37,0.04,0.33,0.46,0.37
3,OpenAI gpt,5.4,1.45,0.73,-0.06,0.04,0.57,NaN,0.45
4,OpenAI gpt,5.5,1.61,0.94,0.22,0.04,0.68,0.69,0.63
5,OpenAI gpt,5.6,1.73,0.95,0.01,0.04,0.76,0.68,0.72
6,Anthropic sonnet,3.5,0.23,1.59,NaN,0.04,NaN,0.41,NaN
7,Anthropic sonnet,4,0.67,1.17,-0.42,0.04,0.03,0.46,NaN
8,Anthropic sonnet,4.5,1.02,1.14,-0.03,0.04,0.07,0.54,0.18
9,Anthropic sonnet,4.6,1.49,0.87,-0.26,0.04,0.59,NaN,0.29
